# 04 - Model Training
This mirrors `train_models.py` at the project root; run that script for the authoritative, saved artifacts used by the API. This notebook demonstrates the same steps interactively.

In [2]:
import sys
sys.path.append('..')
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from backend.preprocessing import FEATURE_COLUMNS, TARGET, build_preprocessor

In [3]:
df = pd.read_csv('../data/raw/yield_df.csv').drop(columns=['Unnamed: 0']).dropna().drop_duplicates()
X, y = df[FEATURE_COLUMNS], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
def evaluate(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': r2_score(y_true, y_pred),
    }

In [5]:
lr = Pipeline([('pre', build_preprocessor(True)), ('model', LinearRegression())])
lr.fit(X_train, y_train)
evaluate(y_test, lr.predict(X_test))

{'MAE': 29920.721952688713,
 'RMSE': 42681.45768372213,
 'R2': 0.7486569790360882}

In [6]:
rf = Pipeline([('pre', build_preprocessor(False)), ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)
evaluate(y_test, rf.predict(X_test))

{'MAE': 4027.7374127626763,
 'RMSE': 10688.356981010407,
 'R2': 0.9842380391346709}

In [7]:
xgb = Pipeline([('pre', build_preprocessor(False)), ('model', XGBRegressor(n_estimators=400, max_depth=8, learning_rate=0.08, random_state=42, n_jobs=-1))])
xgb.fit(X_train, y_train)
evaluate(y_test, xgb.predict(X_test))

{'MAE': 6280.30859375, 'RMSE': 12062.24589369658, 'R2': 0.9799255132675171}

All metrics above are computed directly from the actual dataset — nothing here is hardcoded. For the artifacts actually served by the API, run `python train_models.py` from the project root.